# Project 2 : Arabic Word Embeddings + Text Classification

**Dataset:** Arabic Sentiment Twitter Corpus
https://huggingface.co/datasets/arbml/Arabic_Sentiment_Twitter_Corpus



## Setup & Installations

In [2]:
!pip install -q gensim datasets pyarabic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 15.2 MB/s eta 0:00:00


In [3]:
import re
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from gensim.models import Word2Vec
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# Fix random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

---
## Part 1: Train Arabic Word Embeddings (Word2Vec)

### Step 1 — Load Dataset

In [14]:
dataset = load_dataset("arbml/Arabic_Sentiment_Twitter_Corpus")
print(dataset)

Repo card metadata block was not found. Setting CardData to empty.


DatasetDict({
    train: Dataset({
        features: ['tweet', 'label'],
        num_rows: 47000
    })
    test: Dataset({
        features: ['tweet', 'label'],
        num_rows: 11751
    })
})


In [20]:
train_data = dataset['train']

texts = [item['tweet'] for item in train_data]
labels_raw = [item['label'] for item in train_data]

print(f"Total samples: {len(texts)}")
print(f"Label distribution: {dict(zip(*np.unique(labels_raw, return_counts=True)))}")
print(f"\nExample text: {texts[0]}")

Total samples: 47000
Label distribution: {np.int64(0): np.int64(23121), np.int64(1): np.int64(23879)}

Example text: اعترف ان بتس كانو شوي شوي يجيبو راسي لكن اليوم بالزايد 😭


### Step 2 — Preprocessing

In [21]:
def preprocess_arabic(text):
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Remove mentions and hashtags
    text = re.sub(r'@\w+|#\w+', '', text)
    # Keep only Arabic characters
    text = re.sub(r'[^\u0600-\u06FF\s]', '', text)
    # Normalize alef variants to normal alef
    text = re.sub(r'[أإآ]', 'ا', text)
    # Normalize taa marbuta
    text = re.sub(r'ة', 'ه', text)
    # Remove tashkeel (diacritics)
    text = re.sub(r'[\u064B-\u065F]', '', text)
    # Collapse extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # Tokenize
    tokens = text.split()
    return tokens


# Apply preprocessing
tokenized_corpus = [preprocess_arabic(t) for t in texts]

# Remove empty entries
valid_indices = [i for i, tokens in enumerate(tokenized_corpus) if len(tokens) > 0]
tokenized_corpus = [tokenized_corpus[i] for i in valid_indices]
labels_raw = [labels_raw[i] for i in valid_indices]

print(f"Corpus size after cleaning: {len(tokenized_corpus)} sentences")
print(f"Example tokenized: {tokenized_corpus[0]}")

Corpus size after cleaning: 46976 sentences
Example tokenized: ['اعترف', 'ان', 'بتس', 'كانو', 'شوي', 'شوي', 'يجيبو', 'راسي', 'لكن', 'اليوم', 'بالزايد']


### Step 3 — Train Word2Vec Models

 train two configurations for comparison:
- **Config 1:** Skip-gram
- **Config 2:** CBOW

In [22]:
# Skip-gram
print("Skip-gram model ")
w2v_skipgram = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=100,   # embedding dimensions
    window=5,          # context window size
    min_count=2,       # ignore words appearing less than 2 times
    sg=1,              # 1 = Skip-gram
    workers=4,
    epochs=10,
    seed=42
)

print(f"Vocabulary size: {len(w2v_skipgram.wv)}")
print(f"Vector shape: {w2v_skipgram.wv.vectors.shape}")

Skip-gram model 
Vocabulary size: 30258
Vector shape: (30258, 100)


In [23]:
# CBOW
print("CBOW model")
w2v_cbow = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=100,
    window=5,
    min_count=2,
    sg=0, # 0 = CBOW
    workers=4,
    epochs=10,
    seed=42
)

print(f"Vocabulary size: {len(w2v_cbow.wv)}")

CBOW model
Vocabulary size: 30258


### Step 4 — Evaluate Embeddings (Qualitative)

In [24]:
# Check semantically similar words
test_words = ['جيد', 'سعيد', 'حزين', 'رائع']

print("Skip-gram Most Similar Words")
for word in test_words:
    if word in w2v_skipgram.wv:
        similar = w2v_skipgram.wv.most_similar(word, topn=5)
        print(f"\n'{word}' -> {similar}")
    else:
        print(f"\n'{word}' not in vocabulary")

Skip-gram Most Similar Words

'جيد' -> [('بمقدار', 0.8806806206703186), ('ستمضي', 0.8738144636154175), ('معصيه', 0.8669354915618896), ('فروعنا', 0.8651748299598694), ('فاجانا', 0.8642206192016602)]

'سعيد' -> [('يطنون', 0.763627290725708), ('مسائكم', 0.7631274461746216), ('تنشر', 0.7581415772438049), ('نهاركم', 0.7520074844360352), ('اخواني', 0.7473859190940857)]

'حزين' -> [('وحالتي', 0.8648715615272522), ('خاله', 0.8198915123939514), ('مكسور', 0.8093368411064148), ('مايمكن', 0.8000448942184448), ('يالباث', 0.7716126441955566)]

'رائع' -> [('عالي', 0.8109738826751709), ('تركيز', 0.7762390375137329), ('مستواك', 0.7671236395835876), ('وانقاذ', 0.7663244009017944), ('لخساره', 0.7321687936782837)]


In [25]:
print("CBOW Most Similar Words")
for word in test_words:
    if word in w2v_cbow.wv:
        similar = w2v_cbow.wv.most_similar(word, topn=5)
        print(f"\n'{word}' -> {similar}")
    else:
        print(f"\n'{word}' not in vocabulary")

CBOW Most Similar Words

'جيد' -> [('فهم', 0.8976427912712097), ('الواقع', 0.8971893787384033), ('تسرق', 0.8970261216163635), ('الطريق', 0.8966040015220642), ('الادب', 0.8956677317619324)]

'سعيد' -> [('جمييل', 0.8964691758155823), ('ليوم', 0.8826650977134705), ('لى', 0.8801522254943848), ('احزنني', 0.8766342997550964), ('مشرق', 0.8754999041557312)]

'حزين' -> [('ضاعت', 0.9299464225769043), ('بشعور', 0.9278690218925476), ('مقدر', 0.9267693161964417), ('واجد', 0.9225611090660095), ('اسفه', 0.9212539792060852)]

'رائع' -> [('انتاج', 0.8962891697883606), ('فوائد', 0.8846065402030945), ('وم', 0.8780750036239624), ('عالي', 0.8774022459983826), ('بالدم', 0.8743587136268616)]


In [27]:
# Save both models
w2v_skipgram.save("w2v_skipgram.model")
w2v_cbow.save("w2v_cbow.model")
print("Models saved :)")

Models saved :)


---
## Part 2: Sentence Representation


In [28]:
def sentence_to_vector(tokens, wv_model, vector_size=100):
    word_vectors = []
    for token in tokens:
        if token in wv_model.wv:
            word_vectors.append(wv_model.wv[token])

    if len(word_vectors) == 0:
        return np.zeros(vector_size)

    return np.mean(word_vectors, axis=0)


# Build sentence vectors using Skip-gram embeddings
X_skipgram = np.array([
    sentence_to_vector(tokens, w2v_skipgram)
    for tokens in tokenized_corpus
])

# Build sentence vectors using CBOW embeddings
X_cbow = np.array([
    sentence_to_vector(tokens, w2v_cbow)
    for tokens in tokenized_corpus
])

print(f"Skip-gram sentence matrix shape: {X_skipgram.shape}")
print(f"CBOW sentence matrix shape:      {X_cbow.shape}")

Skip-gram sentence matrix shape: (46976, 100)
CBOW sentence matrix shape:      (46976, 100)


In [29]:
# Encode labels as integers
unique_labels = sorted(set(labels_raw))
label2idx = {lbl: idx for idx, lbl in enumerate(unique_labels)}
y = np.array([label2idx[lbl] for lbl in labels_raw])

print(f"Label mapping: {label2idx}")
print(f"Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}")

Label mapping: {0: 0, 1: 1}
Class distribution: {np.int64(0): np.int64(23104), np.int64(1): np.int64(23872)}


In [30]:
# Train/test split (80/20)
X_sg_train, X_sg_test, y_train, y_test = train_test_split(
    X_skipgram, y, test_size=0.2, random_state=42, stratify=y
)

X_cbow_train, X_cbow_test, _, _ = train_test_split(
    X_cbow, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {len(X_sg_train)}")
print(f"Test size:  {len(X_sg_test)}")

Train size: 37580
Test size:  9396


---
## Part 3: Text Classification with PyTorch

In [32]:
class SentimentDataset(Dataset):
    def __init__(self, vectors, labels):
        self.X = torch.tensor(vectors, dtype=torch.float32)
        self.y = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [33]:
class SentimentClassifier(nn.Module):

    def __init__(self, input_size, hidden_size, num_classes, dropout=0.3):
        super(SentimentClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, num_classes)
        )

    def forward(self, x):
        return self.network(x)

In [37]:
def train_model(X_train, X_test, y_train, y_test, model_name="Model",
                input_size=100, num_classes=2, epochs=30, lr= 0.001, batch_size=64):

    # DataLoaders
    train_loader = DataLoader(
        SentimentDataset(X_train, y_train), batch_size=batch_size, shuffle=True
    )
    test_loader = DataLoader(
        SentimentDataset(X_test, y_test), batch_size=batch_size
    )

    # Model
    model = SentimentClassifier(
        input_size=input_size,
        hidden_size=128,
        num_classes=num_classes
    )

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Training
    print(f"  Training: {model_name}")

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        # Print every 5 epochs
        if (epoch + 1) % 5 == 0:
            avg_loss = total_loss / len(train_loader)
            print(f"  Epoch [{epoch+1:02d}/{epochs}]  Loss: {avg_loss:.4f}")

    # Evaluation
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            outputs = model(X_batch)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.numpy())
            all_labels.extend(y_batch.numpy())

    acc = accuracy_score(all_labels, all_preds)
    print(f"\n  Test Accuracy: {acc:.4f}")
    print(f"\n  Classification Report:")
    print(classification_report(all_labels, all_preds, target_names=[str(k) for k in label2idx.keys()]))

    return acc, model

### Experiment 1: Skip-gram Embeddings

In [38]:
num_classes = len(unique_labels)

acc_sg, model_sg = train_model(
    X_sg_train, X_sg_test, y_train, y_test,
    model_name="Skip-gram (vector_size=100)",
    input_size=100,
    num_classes=num_classes,
    epochs=30
)

  Training: Skip-gram (vector_size=100)
  Epoch [05/30]  Loss: 0.5218
  Epoch [10/30]  Loss: 0.4880
  Epoch [15/30]  Loss: 0.4697
  Epoch [20/30]  Loss: 0.4536
  Epoch [25/30]  Loss: 0.4399
  Epoch [30/30]  Loss: 0.4339

  Test Accuracy: 0.7494

  Classification Report:
              precision    recall  f1-score   support

           0       0.71      0.82      0.76      4621
           1       0.80      0.68      0.73      4775

    accuracy                           0.75      9396
   macro avg       0.75      0.75      0.75      9396
weighted avg       0.76      0.75      0.75      9396



### Experiment 2: CBOW Embeddings

In [39]:
acc_cbow, model_cbow = train_model(
    X_cbow_train, X_cbow_test, y_train, y_test,
    model_name="CBOW (vector_size=100)",
    input_size=100,
    num_classes=num_classes,
    epochs=30
)

  Training: CBOW (vector_size=100)
  Epoch [05/30]  Loss: 0.5697
  Epoch [10/30]  Loss: 0.5425
  Epoch [15/30]  Loss: 0.5311
  Epoch [20/30]  Loss: 0.5193
  Epoch [25/30]  Loss: 0.5120
  Epoch [30/30]  Loss: 0.5044

  Test Accuracy: 0.7209

  Classification Report:
              precision    recall  f1-score   support

           0       0.70      0.77      0.73      4621
           1       0.75      0.68      0.71      4775

    accuracy                           0.72      9396
   macro avg       0.72      0.72      0.72      9396
weighted avg       0.72      0.72      0.72      9396



### Results Comparison

In [46]:
print("EXPERIMENT COMPARISON")
print(" ")
print(f"Skip-gram Accuracy: {acc_sg:.4f}")
print(f"CBOW Accuracy: {acc_cbow:.4f}")


better = "Skip-gram" if acc_sg > acc_cbow else "CBOW"
print(f"\nBetter configuration: {better}")

EXPERIMENT COMPARISON
 
Skip-gram Accuracy: 0.7494
CBOW Accuracy: 0.7209

Better configuration: Skip-gram
